In [3]:
import sys
sys.path.append("..")

In [4]:
import joblib
from pathlib import Path
import numpy as np
import pandas as pd

from src.tokenizer.corpus import Corpus
from src.tokenizer.statistics import Statistics
from src.tokenizer.candidates import CandidateExtractor
from src.tokenizer.vocabulary import VocabularyBuilder
from src.tokenizer.normalizer import Normalizer
from src.tokenizer.vectorizer import VocabularyVectorizer

In [5]:
df = pd.read_csv(
    "../data/text_corpus.csv",
    encoding="utf-8"
)


texts = df["comment"].tolist()


corpus = Corpus(texts)

In [6]:
print(type(corpus))

print(len(corpus))

<class 'src.tokenizer.corpus.Corpus'>
10000


In [7]:
stats = Statistics().fit(corpus)


In [8]:
len(stats.phrase_freq)

1559

In [9]:

normalizer = Normalizer()

extractor = CandidateExtractor(
    normalizer=normalizer,
    min_freq=20
)

candidates = extractor.extract(stats)

builder = VocabularyBuilder()

vocabulary = builder.build(candidates)


In [10]:
len(candidates), len(vocabulary)

(506, 506)

In [11]:

df_vocab = builder.to_dataframe(vocabulary)

df_vocab.head(50)

,text,freq,score,parent_count,child_count,redundancy,rank
0,неисправность оборудования,5343,15112.286128,1,2,1.092638,13298.811792
1,есть проблемы,3586,10142.739669,0,0,0.000000,10142.739669
2,оборудования провайдера,2894,8185.468099,0,3,0.000000,9413.288314
3,проблема оборудования,2621,7413.307494,1,1,0.555297,7783.972869
4,неисправность оборудования провайдера,1479,7685.109433,2,0,0.302454,7685.109433
5,проблема абонентского,2536,7172.891188,1,1,0.537288,7531.535748
6,отсутствует,6159,6159.000000,0,4,0.000000,7390.800000
7,неисправность оборудования абонента,1419,7373.340288,2,0,0.290184,7373.340288
8,проблема оборудования провайдера,1415,7352.555678,2,0,0.539870,7352.555678
9,проблема абонентского оборудования,1382,7181.082648,2,0,0.544953,7181.082648


In [12]:
df_vocab.sort_values(
    "rank",
    ascending=False
).head(50)

,text,freq,score,parent_count,child_count,redundancy,rank
0,неисправность оборудования,5343,15112.286128,1,2,1.092638,13298.811792
1,есть проблемы,3586,10142.739669,0,0,0.000000,10142.739669
2,оборудования провайдера,2894,8185.468099,0,3,0.000000,9413.288314
3,проблема оборудования,2621,7413.307494,1,1,0.555297,7783.972869
4,неисправность оборудования провайдера,1479,7685.109433,2,0,0.302454,7685.109433
5,проблема абонентского,2536,7172.891188,1,1,0.537288,7531.535748
6,отсутствует,6159,6159.000000,0,4,0.000000,7390.800000
7,неисправность оборудования абонента,1419,7373.340288,2,0,0.290184,7373.340288
8,проблема оборудования провайдера,1415,7352.555678,2,0,0.539870,7352.555678
9,проблема абонентского оборудования,1382,7181.082648,2,0,0.544953,7181.082648


In [13]:
df_vocab[
    df_vocab["text"].str.contains(
        "авария|неисправность|оборудование|соединение"
    )
].head(50)

,text,freq,score,parent_count,child_count,redundancy,rank
0,неисправность оборудования,5343,15112.286128,1,2,1.092638,13298.811792
4,неисправность оборудования провайдера,1479,7685.109433,2,0,0.302454,7685.109433
7,неисправность оборудования абонента,1419,7373.340288,2,0,0.290184,7373.340288
14,неисправность,4890,4890.000000,0,3,0.000000,5623.500000
19,авария,3300,3300.000000,0,5,0.000000,4125.000000
22,соединение отсутствует,1308,3699.582679,0,0,0.000000,3699.582679
24,авария на,1203,3402.597831,1,1,0.364545,3572.727723
25,авария сети,1188,3360.171424,1,1,0.360000,3528.179995
27,авария оборудования,1220,3450.681092,1,0,0.369697,3450.681092
29,авария на стороне,658,3419.068294,2,0,0.546966,3419.068294


In [14]:
df_vocab.sort_values("rank", ascending=False).reset_index(drop=True).reset_index(
    ).rename(columns={'index':'token_id'}).to_csv("../data/vocabulary.csv", index=False)

df_vocab = pd.read_csv("../data/vocabulary.csv")
df_vocab.info()

<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   token_id      506 non-null    int64  
 1   text          506 non-null    str    
 2   freq          506 non-null    int64  
 3   score         506 non-null    float64
 4   parent_count  506 non-null    int64  
 5   child_count   506 non-null    int64  
 6   redundancy    506 non-null    float64
 7   rank          506 non-null    float64
dtypes: float64(3), int64(4), str(1)
memory usage: 31.8 KB


In [15]:
len(df_vocab)

506

In [16]:
df_vocab["freq"].describe()

count     506.000000
mean      236.124506
std       642.334334
min        20.000000
25%        30.000000
50%        49.000000
75%       120.750000
max      6159.000000
Name: freq, dtype: float64

In [17]:
vocab_set = set(
    df_vocab["text"]
)

coverage = []

for text in corpus:

    count = 0

    for token in vocab_set:

        if token in text.lower():
            count += 1

    coverage.append(count)

pd.Series(coverage).describe()

count    10000.000000
mean         8.601200
std          2.403734
min          3.000000
25%          7.000000
50%          8.000000
75%         10.000000
max         17.000000
dtype: float64

In [18]:
df_vocab.sort_values(
    "freq"
).head(50)

,token_id,text,freq,score,parent_count,child_count,redundancy,rank
454,454,абонента передано специалистам,20,103.923048,1,0,1.000000,83.138439
455,455,wi-fi проблема абонентского,20,103.923048,2,0,1.000000,83.138439
457,457,pon соединение отсутствует,20,103.923048,2,0,1.000000,83.138439
460,460,wi-fi есть проблемы,20,103.923048,2,0,1.000000,83.138439
459,459,тв соединение отсутствует,20,103.923048,1,0,1.000000,83.138439
458,458,по ожидается восстановление,20,103.923048,1,0,1.000000,83.138439
456,456,legacy есть проблемы,20,103.923048,2,0,1.000000,83.138439
415,415,tdm зарегистрировано соединение,20,103.923048,2,0,0.465116,103.923048
416,416,отсутствует причина неверная,20,103.923048,1,0,0.003247,103.923048
420,420,цифровое тв соединение,20,103.923048,2,0,0.166667,103.923048


In [19]:
vectorizer = VocabularyVectorizer(
    "../data/vocabulary.csv"
)


X = vectorizer.transform(
    corpus
)


X.shape

(10000, 506)

In [20]:
joblib.dump(X, "../data/numpy_X_vectorized.joblib")

['../data/numpy_X_vectorized.joblib']

In [21]:
np.asarray(
    X.sum(axis=1)
).ravel().mean()

np.float64(8.6012)

In [22]:
nnz_per_doc = np.asarray(
    X.sum(axis=1)
).ravel()

pd.Series(nnz_per_doc).describe()

count    10000.000000
mean         8.601200
std          2.403734
min          3.000000
25%          7.000000
50%          8.000000
75%         10.000000
max         17.000000
dtype: float64

In [23]:
(df_vocab[
    df_vocab["text"].str.contains(
        "неисправность",
        na=False
    )
][
    [
        "token_id",
        "text",
        "freq"
    ]
])

,token_id,text,freq
0,0,неисправность оборудования,5343
4,4,неисправность оборудования провайдера,1479
7,7,неисправность оборудования абонента,1419
14,14,неисправность,4890
74,74,причина неисправность оборудования,173
102,102,причина неисправность,173
123,123,отсутствует причина неисправность,79
249,249,fttx неисправность оборудования,43
272,272,проблемы причина неисправность,32
339,339,интернет fttx неисправность,25
